In [143]:
import requests
import xmltodict
import xml.etree.ElementTree as ET
from requests.packages.urllib3.exceptions import InsecureRequestWarning
import ssl
import base64
import sys

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)
ssl_context = ssl.create_default_context(purpose=ssl.Purpose.CLIENT_AUTH)
ssl_context.verify_mode = ssl.CERT_NONE

user=base64.b64decode('aHNjcm9vdA==').decode()
passwd=base64.b64decode('V2VsYzBtZTEyMw==').decode()


def loginHmc(user,passwd,HMCname):
    logonheaders = {'Content-Type': 'application/vnd.ibm.powervm.web+xml; type=LogonRequest'}
    logonUrl =  'https://'+HMCname+':12443/rest/api/web/Logon'
    logonPayload = '<LogonRequest schemaVersion="V1_0" xmlns="http://www.ibm.com/xmlns/systems/power/firmware/web/mc/2012_10/" xmlns:mc="http://www.ibm.com/xmlns/systems/power/firmware/web/mc/2012_10/"><UserID>' + user + '</UserID><Password>' + passwd + '</Password></LogonRequest>'
    proxies = {
                "http": "",
                "https": "",
                }
    ret = requests.put(logonUrl,data=logonPayload,headers=logonheaders,proxies=proxies,verify=False)

    if ret.status_code != 200:
        print("Error: Logon failed error code=%d url=%s" %(ret.status_code,logonUrl))
        if self.debug:
                print("DEBUG: Returned:%s" %(ret.text))
        # Do not return if we failed to logon
        sys.exit(ret.status_code)
    xmlResponse = ET.fromstring(ret.text)
    token = xmlResponse[1].text
    return token

def logoffHmc(hmc,token):
    logoffheaders = {'X-API-Session' : token }
    logoffUrl =  'https://'+hmc+':12443/rest/api/web/Logon'
    proxies = {
                "http": "",
                "https": "",
                }
    ret = requests.delete(logoffUrl,headers=logoffheaders,proxies=proxies,verify=False)
    rcode = ret.status_code
    if rcode == 200 or rcode == 202 or rcode == 204:
        print("DEBUG:Successfully disconnected from the HMC (code=%d)" %(rcode))
    else:
        print("Error: Logoff failed error code=%d url=%s data=%s" %(rcode, logoffUrl, ret.text))
        sys.exit(rcode)

In [144]:
HMCname='10.125.48.40'

token=loginHmc(user,passwd,HMCname)
#print(token)

prefHeaders = {'X-API-Session' : token}
prefUrl = 'https://'+HMCname+'/rest/api/uom/ManagedSystem'
proxies = {
                "http": "",
                "https": "",
                }
ret = requests.get(prefUrl, headers=prefHeaders,proxies=proxies,verify=False).content#.decode('utf-8')
#print(ret)
out=xmltodict.parse(ret)

In [145]:
for lpars in range(0,len(out.get('feed').get('entry'))):
    print(out.get('feed').get('entry')[lpars].get('content'))
    print('=================================================================')

{'@type': 'application/vnd.ibm.powervm.uom+xml; type=ManagedSystem', 'ManagedSystem:ManagedSystem': {'@xmlns:ManagedSystem': 'http://www.ibm.com/xmlns/systems/power/firmware/uom/mc/2012_10/', '@xmlns': 'http://www.ibm.com/xmlns/systems/power/firmware/uom/mc/2012_10/', '@xmlns:ns2': 'http://www.w3.org/XML/1998/namespace/k2', '@schemaVersion': 'V1_0', 'Metadata': {'Atom': {'AtomID': '4e76819f-24f4-3167-b8b5-212d0d412c47', 'AtomCreated': '1716302010308'}}, 'ActivatedLevel': {'@kxe': 'false', '@kb': 'ROR', '#text': '131'}, 'ActivatedServicePackNameAndLevel': {'@ksv': 'V1_9_0', '@kxe': 'false', '@kb': 'ROR', '#text': 'FW950.80 (131)'}, 'AssociatedIPLConfiguration': {'@group': 'Hypervisor', '@kb': 'CUD', '@kxe': 'false', '@schemaVersion': 'V1_0', 'Metadata': {'Atom': None}, 'CurrentManufacturingDefaulConfigurationtBootMode': {'@kxe': 'false', '@kb': 'ROR', '#text': 'Normal'}, 'CurrentPowerOnSide': {'@kb': 'ROR', '@kxe': 'false', '#text': 'Temporary'}, 'CurrentSystemKeylock': {'@kxe': 'false'

In [146]:
print(out.get('feed').get('entry')[4].get('content').get('ManagedSystem:ManagedSystem').get('SystemName').get('#text'))
print(out.get('feed').get('entry')[4].get('content').get('ManagedSystem:ManagedSystem').get('Metadata').get('Atom').get('AtomID'))

DCMIPPHPWR002-SN7837B88
5aa8e961-a455-3184-8b79-9579571179f5


In [147]:

c

In [160]:
#print(out2)

In [149]:
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('SystemName').get('#text'))
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('PartitionID').get('#text'))
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('PartitionName').get('#text'))
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('ResourceMonitoringIPAddress').get('#text'))
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('OperatingSystemType').get('#text'))
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('OperatingSystemVersion').get('#text'))
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('PartitionState').get('#text'))
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('PartitionUUID').get('#text'))
print(out2.get('feed').get('entry')[4].get('content').get('LogicalPartition:LogicalPartition').get('CurrentProcessorCompatibilityMode').get('#text'))

DCMIPPHPWR002-SN7837B88
20
d1mipvmfwt001
10.125.48.219
AIX
AIX 7.2 7200-05-07-2346
running
6B734364-1B89-415B-A69D-A3881FAB5B42
POWER8


In [150]:
prefHeaders = {'X-API-Session' : token}
prefUrl = 'https://'+HMCname+'/rest/api/uom/ManagedSystem/5aa8e961-a455-3184-8b79-9579571179f5/LogicalPartition/593E86F3-74E4-4FB4-BD7D-806403CD9DEE/VirtualFibreChannelClientAdapter'
proxies = {
                "http": "",
                "https": "",
                }
ret = requests.get(prefUrl, headers=prefHeaders,proxies=proxies,verify=False).content#.decode('utf-8')
#print(ret)
out3=xmltodict.parse(ret)

In [159]:
for wwpns in range(0,len(out3.get('feed').get('entry'))):
    print(out3.get('feed').get('entry')[wwpns].get('content').get('VirtualFibreChannelClientAdapter:VirtualFibreChannelClientAdapter').get('WWPNs').get('#text'))
    print("=="*20)

c050760b3c4e0024 c050760b3c4e0025
c050760b3c4e0026 c050760b3c4e0027


None


In [ ]:
logoffHmc(HMCname,token)